# TatVTON — ControlNet Training Pipeline

End-to-end pipeline for training `khope/tatvton-controlnet-v1`:

| Phase | Task | GPU Required |
|-------|------|--------------|
| 1 | Upload crawled images to Colab | No |
| 2 | DensePose IUV map generation | Yes |
| 3 | Build HuggingFace Dataset | No |
| 4 | ControlNet SDXL Training | Yes (A100) |
| 5 | Push trained model to HF Hub | No |

**Runtime**: A100 GPU recommended (Colab Pro+)

## Phase 0: Setup & Dependencies

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q diffusers transformers accelerate datasets huggingface-hub safetensors
!pip install -q Pillow numpy

# DensePose (detectron2)
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# Clone TatVTON repo
!git clone https://github.com/art8engine/TatVTON.git /content/TatVTON 2>/dev/null || true
%cd /content/TatVTON

In [ ]:
# HuggingFace login (needed for push)
from huggingface_hub import notebook_login
notebook_login()

## Phase 1: Download Images from HuggingFace Hub

Download crawled images from `rlaope/tatvton-tattoo-raw`.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

REPO_ID = "rlaope/tatvton-tattoo-raw"
LOCAL_DIR = Path("/content/dataset")
LOCAL_IMAGES = LOCAL_DIR / "images"
LOCAL_CONDITIONING = LOCAL_DIR / "conditioning"
LOCAL_CONDITIONING.mkdir(parents=True, exist_ok=True)

print(f"Downloading {REPO_ID} from HuggingFace Hub...")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=str(LOCAL_DIR),
)

n_images = len(list(LOCAL_IMAGES.glob("*.jpg")))
print(f"Downloaded {n_images} images to {LOCAL_IMAGES}")

In [ ]:
# Load metadata
import json

LOCAL_META = LOCAL_DIR / "metadata_crawled.json"

if LOCAL_META.exists():
    with open(LOCAL_META) as f:
        meta = json.load(f)
    print(f"Loaded metadata: {len(meta)} entries")
else:
    print("No metadata found. Will generate captions from scratch.")

n_images = len(list(LOCAL_IMAGES.glob("*.jpg")))
print(f"Total images ready: {n_images}")

## Phase 2: DensePose IUV Map Generation

Runs DensePose on every image to generate:
- IUV conditioning maps (for ControlNet)
- Body part labels (from I channel)

In [ ]:
# Install DensePose specifically
!pip install -q 'git+https://github.com/facebookresearch/detectron2@main#subdirectory=projects/DensePose'

In [ ]:
import numpy as np
from PIL import Image
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

try:
    from densepose import add_densepose_config
    from densepose.structures import DensePoseChartPredictorOutput
    DENSEPOSE_OK = True
    print("DensePose loaded successfully")
except ImportError:
    DENSEPOSE_OK = False
    print("DensePose import failed — check installation")

In [ ]:
# DensePose body part mapping
# I channel values → body part names
BODY_PART_MAP = {
    0: "background",
    1: "torso_back", 2: "torso_front",
    3: "right_hand", 4: "left_hand",
    5: "left_foot", 6: "right_foot",
    7: "upper_leg_right_back", 8: "upper_leg_left_back",
    9: "upper_leg_right_front", 10: "upper_leg_left_front",
    11: "lower_leg_right_back", 12: "lower_leg_left_back",
    13: "lower_leg_right_front", 14: "lower_leg_left_front",
    15: "upper_arm_left_back", 16: "upper_arm_right_back",
    17: "upper_arm_left_front", 18: "upper_arm_right_front",
    19: "lower_arm_left_back", 20: "lower_arm_right_back",
    21: "lower_arm_left_front", 22: "lower_arm_right_front",
    23: "head_right", 24: "head_left",
}

# Simplified mapping for our 6 categories
I_TO_CATEGORY = {
    0: None,  # background
    1: "back", 2: "chest",
    3: "arm", 4: "arm",
    5: "leg", 6: "leg",
    7: "leg", 8: "leg", 9: "leg", 10: "leg",
    11: "leg", 12: "leg", 13: "leg", 14: "leg",
    15: "arm", 16: "arm", 17: "arm", 18: "arm",
    19: "arm", 20: "arm", 21: "arm", 22: "arm",
    23: "neck", 24: "neck",  # head ≈ neck area for tattoos
}

def dominant_body_part(iuv: np.ndarray) -> str:
    """Get the dominant body part from IUV map's I channel."""
    i_channel = iuv[:, :, 0]
    # Ignore background (0)
    nonzero = i_channel[i_channel > 0]
    if len(nonzero) == 0:
        return "unknown"
    
    from collections import Counter
    counts = Counter()
    for val in nonzero:
        cat = I_TO_CATEGORY.get(int(val))
        if cat:
            counts[cat] += 1
    
    if not counts:
        return "unknown"
    return counts.most_common(1)[0][0]

print("Body part classifier ready")

In [ ]:
# Build DensePose predictor
import os
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from densepose import add_densepose_config

cfg = get_cfg()
add_densepose_config(cfg)

# Download config + weights
DENSEPOSE_CFG_URL = "https://raw.githubusercontent.com/facebookresearch/detectron2/main/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml"
DENSEPOSE_WEIGHTS_URL = "https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl"

cfg_path = "/content/densepose_rcnn_R_50_FPN_s1x.yaml"
if not os.path.exists(cfg_path):
    !wget -q {DENSEPOSE_CFG_URL} -O {cfg_path}

cfg.merge_from_file(cfg_path)
cfg.MODEL.WEIGHTS = DENSEPOSE_WEIGHTS_URL
cfg.MODEL.DEVICE = "cuda"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)
print("DensePose predictor ready")

In [ ]:
def extract_iuv(predictor, image_bgr: np.ndarray) -> np.ndarray:
    """Run DensePose, return (H, W, 3) IUV map."""
    h, w = image_bgr.shape[:2]
    iuv = np.zeros((h, w, 3), dtype=np.uint8)

    outputs = predictor(image_bgr)
    instances = outputs["instances"]

    if not instances.has("pred_densepose"):
        return iuv

    dp = instances.pred_densepose
    boxes = instances.pred_boxes.tensor.cpu().numpy()

    for i in range(len(dp)):
        x1, y1, x2, y2 = boxes[i].astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        bh, bw = y2 - y1, x2 - x1
        if bh <= 0 or bw <= 0:
            continue

        result_i = dp[i].labels.cpu().numpy().astype(np.uint8)
        result_u = (dp[i].uv[0].cpu().numpy() * 255).astype(np.uint8)
        result_v = (dp[i].uv[1].cpu().numpy() * 255).astype(np.uint8)

        i_r = np.array(Image.fromarray(result_i).resize((bw, bh), Image.NEAREST))
        u_r = np.array(Image.fromarray(result_u).resize((bw, bh), Image.BILINEAR))
        v_r = np.array(Image.fromarray(result_v).resize((bw, bh), Image.BILINEAR))

        iuv[y1:y2, x1:x2, 0] = i_r
        iuv[y1:y2, x1:x2, 1] = u_r
        iuv[y1:y2, x1:x2, 2] = v_r

    return iuv

print("IUV extraction function ready")

In [ ]:
# Batch process all images
import json
from pathlib import Path

LOCAL_IMAGES = Path("/content/dataset/images")
LOCAL_CONDITIONING = Path("/content/dataset/conditioning")
LOCAL_CONDITIONING.mkdir(parents=True, exist_ok=True)

image_paths = sorted(LOCAL_IMAGES.glob("*.jpg"))
print(f"Processing {len(image_paths)} images...")

results = []
failed = 0

for idx, img_path in enumerate(image_paths):
    out_path = LOCAL_CONDITIONING / f"{img_path.stem}.png"
    
    if out_path.exists():
        continue
    
    try:
        img = Image.open(img_path).convert("RGB")
        img_bgr = np.array(img)[:, :, ::-1]  # RGB → BGR
        
        iuv = extract_iuv(predictor, img_bgr)
        has_person = iuv.sum() > 0
        body_part = dominant_body_part(iuv) if has_person else "unknown"
        
        Image.fromarray(iuv, "RGB").save(out_path)
        
        results.append({
            "filename": img_path.name,
            "iuv_file": out_path.name,
            "has_person": has_person,
            "body_part": body_part,
        })
    except Exception as e:
        print(f"Failed: {img_path.name}: {e}")
        failed += 1
    
    if (idx + 1) % 100 == 0:
        print(f"Progress: {idx+1}/{len(image_paths)} (failed={failed})")

# Save results
with open("/content/dataset/metadata_iuv.json", "w") as f:
    json.dump(results, f, indent=2)

has_person_count = sum(1 for r in results if r["has_person"])
print(f"\nDone: {len(results)} IUV maps ({has_person_count} with person)")
print(f"Failed: {failed}")

# Body part distribution
from collections import Counter
dist = Counter(r["body_part"] for r in results)
print("\nBody part distribution (DensePose):")
for part, count in dist.most_common():
    print(f"  {part:20s} {count:4d}")

In [ ]:
# Visualize a few IUV maps
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
samples = sorted(LOCAL_CONDITIONING.glob("*.png"))[:4]

for i, iuv_path in enumerate(samples):
    img_path = LOCAL_IMAGES / f"{iuv_path.stem}.jpg"
    img = Image.open(img_path)
    iuv = Image.open(iuv_path)
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(iuv_path.stem[:20])
    axes[0, i].axis("off")
    
    axes[1, i].imshow(iuv)
    axes[1, i].set_title("IUV Map")
    axes[1, i].axis("off")

plt.suptitle("Original (top) vs DensePose IUV (bottom)")
plt.tight_layout()
plt.show()

## Phase 3: Build & Upload HuggingFace Dataset

In [ ]:
import json
from pathlib import Path
from datasets import Dataset, DatasetDict, Features, Value
from datasets import Image as HFImage
from PIL import Image

LOCAL_IMAGES = Path("/content/dataset/images")
LOCAL_CONDITIONING = Path("/content/dataset/conditioning")

# Load metadata
meta_crawled = {}
crawl_path = Path("/content/dataset/metadata_crawled.json")
if crawl_path.exists():
    with open(crawl_path) as f:
        for entry in json.load(f):
            meta_crawled[entry["filename"]] = entry

meta_iuv = {}
iuv_path = Path("/content/dataset/metadata_iuv.json")
if iuv_path.exists():
    with open(iuv_path) as f:
        for entry in json.load(f):
            meta_iuv[entry["filename"]] = entry

print(f"Crawl metadata: {len(meta_crawled)}")
print(f"IUV metadata: {len(meta_iuv)}")

In [ ]:
# Build paired dataset
data = {"image": [], "conditioning_image": [], "text": []}
skipped = 0

for img_path in sorted(LOCAL_IMAGES.glob("*.jpg")):
    cond_path = LOCAL_CONDITIONING / f"{img_path.stem}.png"
    if not cond_path.exists():
        skipped += 1
        continue
    
    # Check if person was detected
    iuv_info = meta_iuv.get(img_path.name, {})
    if not iuv_info.get("has_person", False):
        skipped += 1
        continue
    
    # Build caption
    crawl_info = meta_crawled.get(img_path.name, {})
    body_part = iuv_info.get("body_part", "skin")
    style = crawl_info.get("style_clip", "")
    
    if style:
        caption = f"a photorealistic {style} tattoo on {body_part}, high quality, detailed skin texture"
    else:
        caption = f"a photorealistic tattoo on {body_part}, high quality, detailed skin texture"
    
    try:
        img = Image.open(img_path).convert("RGB")
        cond = Image.open(cond_path).convert("RGB")
        data["image"].append(img)
        data["conditioning_image"].append(cond)
        data["text"].append(caption)
    except Exception:
        skipped += 1

print(f"Paired: {len(data['image'])} images")
print(f"Skipped: {skipped}")

In [ ]:
# Create HF Dataset with train/val split
features = Features({
    "image": HFImage(),
    "conditioning_image": HFImage(),
    "text": Value("string"),
})

total = len(data["image"])
split_idx = int(total * 0.9)

ds_dict = DatasetDict({
    "train": Dataset.from_dict(
        {k: v[:split_idx] for k, v in data.items()}, features=features
    ),
    "validation": Dataset.from_dict(
        {k: v[split_idx:] for k, v in data.items()}, features=features
    ),
})

print(f"Train: {len(ds_dict['train'])}, Validation: {len(ds_dict['validation'])}")
print(f"\nSample caption: {ds_dict['train'][0]['text']}")

In [ ]:
# Push to HuggingFace Hub
DATASET_REPO = "khope/tatvton-dataset-v1"

ds_dict.push_to_hub(DATASET_REPO, private=True)
print(f"Uploaded to https://huggingface.co/datasets/{DATASET_REPO}")

## Phase 4: ControlNet SDXL Training

Uses diffusers' official training script.

**Requirements**: A100 GPU, ~2-3 hours for 3 epochs

In [ ]:
# Clone diffusers and install training deps
!git clone --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers 2>/dev/null || true
!pip install -q -r /content/diffusers/examples/controlnet/requirements_sdxl.txt
!pip install -q wandb

In [ ]:
# Training config
DATASET_REPO = "khope/tatvton-dataset-v1"
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
OUTPUT_DIR = "/content/checkpoints/tatvton-controlnet-v1"

# Hyperparameters
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 4          # effective batch = 4
LR = 1e-5
RESOLUTION = 1024
SAVE_STEPS = 500
VAL_STEPS = 200
SEED = 42

print(f"Dataset: {DATASET_REPO}")
print(f"Base model: {BASE_MODEL}")
print(f"Output: {OUTPUT_DIR}")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}x{GRAD_ACCUM}, LR: {LR}")

In [ ]:
# Launch training
!accelerate launch /content/diffusers/examples/controlnet/train_controlnet_sdxl.py \
    --pretrained_model_name_or_path={BASE_MODEL} \
    --output_dir={OUTPUT_DIR} \
    --dataset_name={DATASET_REPO} \
    --image_column="image" \
    --conditioning_image_column="conditioning_image" \
    --caption_column="text" \
    --resolution={RESOLUTION} \
    --learning_rate={LR} \
    --train_batch_size={BATCH_SIZE} \
    --gradient_accumulation_steps={GRAD_ACCUM} \
    --num_train_epochs={EPOCHS} \
    --mixed_precision="fp16" \
    --gradient_checkpointing \
    --checkpointing_steps={SAVE_STEPS} \
    --validation_steps={VAL_STEPS} \
    --seed={SEED} \
    --report_to="wandb" \
    --tracker_project_name="tatvton-controlnet"

## Phase 5: Push Trained Model to HF Hub

In [ ]:
from diffusers import ControlNetModel

MODEL_REPO = "khope/tatvton-controlnet-v1"
OUTPUT_DIR = "/content/checkpoints/tatvton-controlnet-v1"

print(f"Loading model from {OUTPUT_DIR}...")
model = ControlNetModel.from_pretrained(OUTPUT_DIR)

print(f"Pushing to {MODEL_REPO}...")
model.push_to_hub(MODEL_REPO)

print(f"Done! Model: https://huggingface.co/{MODEL_REPO}")

## Phase 6: Quick Test

Verify the trained model works with TatVTON pipeline.

In [ ]:
# Test the trained ControlNet with TatVTON
!pip install -e /content/TatVTON

from tatvton import TatVTONConfig, TatVTONPipeline, PointPrompt
from PIL import Image

config = TatVTONConfig(
    controlnet_model_id=MODEL_REPO,
    device="cuda",
    offload_strategy="model",
)

pipe = TatVTONPipeline(config)

# Use a sample image
body = Image.open("/content/TatVTON/assets/sample_tattoo.jpg").convert("RGB")

# Simple test with a region
region = PointPrompt(coords=[(body.width // 2, body.height // 2)])

print("Running inference...")
# Note: need a tattoo design image too
# result = pipe(body_image=body, tattoo_image=tattoo, region=region)
print("Pipeline loaded successfully — ready for inference")

pipe.unload()

---

## Summary

After running this notebook:

1. `khope/tatvton-dataset-v1` — HF Hub dataset (image + IUV conditioning + text)
2. `khope/tatvton-controlnet-v1` — Trained ControlNet model on HF Hub
3. TatVTON SDK can now use the trained model end-to-end